In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch06-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "06-05-virtual-screening.py"

# BACE 活性分子會排在前面嗎？
使用 MoleculeNet BACE；第一次執行需連網下載 bace.csv。執行下格後，最後一格會在獨立的 Python 程序計算並畫圖。小練習可以接在寫入程式的儲存格末尾，再重跑該格與最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import numpy as np
import deepchem as dc
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
tasks, (dataset,), _ = dc.molnet.load_bace_regression(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None, transformers=[],
    reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
assert tasks == ["pIC50"]
smiles = [Chem.MolToSmiles(Chem.MolFromSmiles(s)) for s in dataset.ids]
generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
from collections import defaultdict
from sklearn.metrics import roc_curve, roc_auc_score

measurements = defaultdict(list)
for smi, value, weight in zip(smiles, dataset.y[:, 0], dataset.w[:, 0]):
    if weight and np.isfinite(value):
        measurements[smi].append(value)
potency = {s: np.median(v) for s, v in measurements.items()}
query = next(s for s in potency if potency[s] > 7)
library = sorted(set(potency) - {query})
labels = np.array([potency[s] > 7 for s in library])
def fingerprint(smi):
    return generator.GetFingerprint(Chem.MolFromSmiles(smi))

fps = [fingerprint(s) for s in library]
scores = np.array(DataStructs.BulkTanimotoSimilarity(fingerprint(query), fps))
order = np.argsort(-scores, kind="stable")
k = int(np.ceil(0.10 * len(library)))
hits = labels[order[:k]].sum()
ef = (hits / k) / labels.mean()
auc = roc_auc_score(labels, scores)
print("query, pIC50:", query, potency[query])
print("rows, N, actives, k, hits, EF, AUC:", len(dataset), len(library), labels.sum(), k, hits, ef, auc)
for rank, i in enumerate(order[:3], 1):
    print(rank, library[i], round(scores[i], 3), round(potency[library[i]], 3))
fpr, tpr, _ = roc_curve(labels, scores)
fig, axes = plt.subplots(1, 2, figsize=(8, 3.7), layout="constrained")
axes[0].plot(np.arange(len(library)+1)/len(library), np.r_[0, np.cumsum(labels[order])]/labels.sum())
axes[0].scatter(k/len(library), hits/labels.sum(), color="#aa5211", zorder=3, label=f"First {k} compounds")
axes[0].legend()
axes[0].set(xlabel="Fraction of library tested", ylabel="Fraction of actives recovered")
axes[1].plot(fpr, tpr, label=f"ROC AUC = {auc:.3f}")
axes[1].legend()
axes[1].set(xlabel="False positive rate", ylabel="True positive rate")
for ax in axes:
    ax.plot([0, 1], [0, 1], "--", color="grey")
    ax.set(xlim=(0, 1), ylim=(0, 1))
fig.savefig(figure_dir / "06-05-virtual-screening.png", dpi=160)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "06-05-virtual-screening.png")))